# Why PM5 has low accuracy

In this case study, we will take a look at PM5, which has the lowest accuracy across all criterions. Here is the statistics for PM5:

| CRITERION | SUPPORT | ACCURACY | PRECISION | RECALL | F1 |
| -:| -:| -:| -:| -:| -:|
| PM5 | 38 |  0.816 | 0.417 | 1.000 |  0.588 |

We notice that PM5 has a very bad precision but perfect recall. This implies that we successfully identifies all PM5 that should applies, but we are being too aggressive on it and falsely classify "not applied" as "applied". Let's take a look at the confusion matrix to verifies our observation. 

In [1]:
from sklearn.metrics import confusion_matrix, accuracy_score
from pathlib import Path
import sys
import json
import pandas as pd

APPLIES_LABELS = [False, True]

# ----------------- LOAD BS3 SAMPLE ----------------- #
project_root = Path.cwd().parents[1]
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))
from evaluation.question1 import load_gold_answer_df
from pipeline import _safe_variant_name

gold_csv = project_root / "full_evaluation_dataset.csv"

gold_df = load_gold_answer_df(str(gold_csv))

predictions = []
prediction_dir = project_root / "outputs/batch_20260708_132813"

if not prediction_dir.exists():
    raise FileNotFoundError(
        f"Could not found predictions at {prediction_dir}"
    )

non_empty_count = 0
predictions = []
for file in sorted(prediction_dir.glob("*.json")):
    if file.name == "batch_summary.json":
        continue

    with open(file, "r", encoding="utf-8") as f:
        payload = json.load(f)

    variant = payload.get("variant")
    results = payload.get("results")

    has_pm5 = any(
        result.get("criterion") == "PM5" for result in results 
    )

    if not has_pm5:
        continue

    if not variant:
        continue
    variant = _safe_variant_name(variant)

    if not gold_df.loc[variant, "gold_applied"]:
        continue

    gold_answer = "PM5" in gold_df.loc[variant, "gold_applied"]

    pm5_prediction = next(
        (result for result in results if result.get("criterion") == "PM5"),
        None
    )

    if not pm5_prediction:
        continue

    pred_answer = pm5_prediction.get("applies")

    if not isinstance(pred_answer, bool):
        continue

    prediction = {
        "variant": variant,
        "criterion": "PM5",
        "evidence": pm5_prediction.get("evidence", "unknown"),
        "reasoning": pm5_prediction.get("reasoning", "unknown"),
        "tool_used": pm5_prediction.get("tool_used", "unknown"),
        "prediction": pred_answer,
        "gold_answer": gold_answer
    }

    predictions.append(prediction)

# ----------------- COMPUTE CONFUSION MATRIX ----------------- #
pred_df = pd.DataFrame(predictions)
y_true = pred_df["gold_answer"]
y_pred = pred_df["prediction"]
cm = confusion_matrix(
    y_true=y_true,
    y_pred=y_pred,
    labels=APPLIES_LABELS,
)

print("Confusion Matrix (rows=actual applies, cols=predicted applies):")
print("-" * 60)
cm_df = pd.DataFrame(
    cm,
    index=[f"Actual {label}" for label in APPLIES_LABELS],
    columns=[f"Pred {label}" for label in APPLIES_LABELS],
)
print(cm_df.to_string())
print("-" * 60)


Confusion Matrix (rows=actual applies, cols=predicted applies):
------------------------------------------------------------
              Pred False  Pred True
Actual False          26          7
Actual True            0          5
------------------------------------------------------------


In [6]:
incorrect_predictions = pred_df[pred_df["gold_answer"] != pred_df["prediction"]]
incorrect_predictions = incorrect_predictions.to_string()
print(f"{"-" * 100}")
print("INCORRECT PREDICTIONS")
print(incorrect_predictions)
print(f"{"-" * 100}")

----------------------------------------------------------------------------------------------------
INCORRECT PREDICTIONS
                     variant criterion                                                                                                                                                                                                                                                                                                                                                                         evidence                                                                                                                                                                                                                                                                                                                                                                                                                                                                            reasoning 

Take a look at the first variant (NM_000020.3:c.1232G>A). I write the script to retrieve what the variant actually receives and tries to find potential causes of the error.

In [ ]:
from pathlib import Path
from unittest.mock import patch
import json
import sys


def find_code_root(start: Path) -> Path:
    """
    Find the folder containing agents/, tools/, data/, and config.py.

    Supports both repository layouts:
    1. repo_root/demo/agents/task_agent.py
    2. repo_root/agents/task_agent.py
    """
    for directory in [start, *start.parents]:
        demo_dir = directory / "demo"

        if (demo_dir / "agents" / "task_agent.py").exists():
            return demo_dir

        if (
            (directory / "agents" / "task_agent.py").exists()
            and (directory / "data" / "planrag.py").exists()
        ):
            return directory

    raise FileNotFoundError(
        "Could not find the project folder containing agents/task_agent.py"
    )

code_root = find_code_root(Path.cwd())

if str(code_root) not in sys.path:
    sys.path.insert(0, str(code_root))


import agents.task_agent as task_agent
from data.planrag import query

variant = "NM_000020.3:c.1232G>A"
criterion = "PM5"
gene = "ACVRL1"
disease = "HHT"

rag_entry = query(
    criterion,
    gene=gene,
    disease=disease,
)

if rag_entry is None:
    raise RuntimeError(
        f"No PlanRAG entry was found for criterion={criterion}, "
        f"gene={gene}, disease={disease}"
    )

tool_decision = {
    "tool": rag_entry["tool"],
    "reason": "specified by Plan agent",
}

known_vep_annotation = {
    "variant_consequence": "missense_variant",
    "codon_position": 411,
    "amino_acid_ref": "Arg",
    "amino_acid_alt": "Gln",
    "protein_change": "p.Arg411Gln",
    "protein_change_1letter": "p.R411Q",
}


with patch.object(
    task_agent,
    "annotate_variant",
    return_value=known_vep_annotation,
):
    evidence, actual_input = task_agent.run_tool(
        tool_decision=tool_decision,
        variant=variant,
        rag_entry=rag_entry,
        gene=gene,
        disease=disease,
    )


print("Tool selected:")
print(tool_decision["tool"])

print("\nActual input reported by run_tool:")
print(actual_input)

print("\nEvidence returned by run_tool:")
print(json.dumps(evidence, indent=2))

print("\nPM5 instructions passed to the model:")
print(rag_entry["instructions"])




Tool selected:
erepo

Actual input reported by run_tool:
ACVRL1 position 411

Evidence returned by run_tool:
{
  "gene": "ACVRL1",
  "codon_position": 411,
  "classifications": [
    {
      "hgvs": "NM_000020.3(ACVRL1):c.1232G>C (p.Arg411Pro)",
      "protein_change": "p.Arg411Pro",
      "classification": "Pathogenic"
    },
    {
      "hgvs": "NM_000020.3(ACVRL1):c.1231C>T (p.Arg411Trp)",
      "protein_change": "p.Arg411Trp",
      "classification": "Pathogenic/Likely pathogenic"
    }
  ],
  "source": "clinvar",
  "same_aa_classifications": [],
  "different_aa_classifications": [
    {
      "hgvs": "NM_000020.3(ACVRL1):c.1232G>C (p.Arg411Pro)",
      "protein_change": "p.Arg411Pro",
      "classification": "Pathogenic"
    },
    {
      "hgvs": "NM_000020.3(ACVRL1):c.1231C>T (p.Arg411Trp)",
      "protein_change": "p.Arg411Trp",
      "classification": "Pathogenic/Likely pathogenic"
    }
  ]
}

PM5 instructions passed to the model:
The tool returns two pre-split lists:
  diffe

Everthing looks normal. However, one thing caught my attention. In the instruction that we've given to the agent, and also in the original vcep specification, an important principle is 
> reference variants must be classified per HHT VCEP rules, not just any ClinVar submission.

However,in the agents.task_agent.run_tool function, we have a fallback logic for erepo (which is the tool used to classify PM5)

```python
  elif tool == "erepo":
        if not gene:
            return {"error": "Gene symbol could not be determined for ERepo lookup — add this gene to GENE_DB in planrag.py"}, input_value
        gene_label = gene
        vep_result = annotate_variant(input_value)
        if "error" in vep_result:
            return vep_result, input_value
        codon_position = vep_result.get("codon_position")
        if codon_position is None:
            return {"error": "VEP did not return a protein position — cannot search ERepo"}, input_value
        result = search_erepo_by_position(gene_label, codon_position)
        # Remove the query variant itself from results — PS1/PM5 require other variants
        # at the same position. Match on the cdna change (e.g. "c.557G>T").
        query_cdna = input_value.split(":")[-1] if ":" in input_value else None
        if query_cdna and "classifications" in result:
            before = len(result["classifications"])
            result["classifications"] = [
                c for c in result["classifications"]
                if query_cdna not in c.get("hgvs", "")
            ]
            after = len(result["classifications"])
            # print(f"DEBUG - erepo query: {gene_label} position {codon_position} | classifications found: {after} (filtered {before - after} self-match)")
        else:
            # print(f"DEBUG - erepo query: {gene_label} position {codon_position} | classifications found: {len(result.get('classifications', []))}")
            pass

        # ClinVar fallback — if ERepo has no other variants at this codon,
        # search ClinVar for HHT VCEP-classified LP/P variants at the same position.
        if not result.get("classifications"):
            ref_aa = vep_result.get("amino_acid_ref")
            if ref_aa:
                # print(f"DEBUG - erepo: 0 results, falling back to ClinVar for {ref_aa}{codon_position}")
                clinvar_result = search_clinvar_for_codon(
                    gene_label, codon_position, ref_aa, query_cdna
                )
                if "error" not in clinvar_result:
                    # print(f"DEBUG - clinvar fallback: {len(clinvar_result.get('classifications', []))} LP/P variants found at codon {codon_position}")
                    result = clinvar_result
                else:
                    # print(f"DEBUG - clinvar fallback error: {clinvar_result['error']}")
                    pass

        # Split results by alt amino acid — PS1 needs same AA, PM5 needs different AA.
        # Done here deterministically so the LLM never has to reason about it.
        _alt_re = re.compile(r'p\.[A-Za-z]{3}\d+([A-Za-z]{3})')
        alt_aa = vep_result.get("amino_acid_alt", "").lower()
        if alt_aa and "classifications" in result:
            same_aa, diff_aa = [], []
            for c in result["classifications"]:
                m = _alt_re.match(c.get("protein_change", ""))
                if m and m.group(1).lower() == alt_aa:
                    same_aa.append(c)
                else:
                    diff_aa.append(c)
            result["same_aa_classifications"] = same_aa
            result["different_aa_classifications"] = diff_aa
            # print(
            #     f"DEBUG - erepo AA split: same_aa={len(same_aa)} (PS1), "
            #     f"different_aa={len(diff_aa)} (PM5)"
            # )

        return result, f"{gene_label} position {codon_position}"
```

Note that in the original script , if erepo is unavilable, we fall back to using clinvar. However, that is not consistent with the original vcep specificatyion. In other words, our current logic is **No other qualifying HHT-VCEP variant found in ERepo** -> **run_tool falls back to ClinVar** -> **Aggregate ClinVar P/LP variant is returned** -> **It is placed in different_aa_classifications** -> **It is placed in different_aa_classifications** -> **The model follows its instructions and applies PM5**, but maybe the correct logic is we should stop when there is no other qualifying HHT-VCEP variant found in Erepo.

## Tentative Conclusion
We should delete the erepo fallback logic. <- this needs further investigation